In [ ]:
# CELL 1 : Install dependencies (if not installed)
!pip install -q torch torchvision pennylane tensorflow scikit-image medmnist

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pennylane as qml
from skimage.transform import resize
from torchvision import transforms
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import (precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score)
import time
from collections import defaultdict
import json
from pennylane import math
import os
from pathlib import Path

# MedMNIST
import medmnist
from medmnist import BloodMNIST, INFO

DATA_FLAG = "bloodmnist"

_meta = INFO[DATA_FLAG]
if "n_classes" in _meta:
    NUM_CLASSES = int(_meta["n_classes"])
else:
    labels = _meta.get("label", [])
    NUM_CLASSES = len(labels) if hasattr(labels, "__len__") else int(_meta.get("n_class", 0) or 0)
    if NUM_CLASSES == 0:
        NUM_CLASSES = 8  # safe fallback for BloodMNIST

print(f"{DATA_FLAG} → NUM_CLASSES = {NUM_CLASSES}")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
# CELL 2 :
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def extract_patches(image, patch_size=3, stride=2):
    patches = image.unfold(2, patch_size, stride).unfold(3, patch_size, stride)
    patches = patches.contiguous().view(image.shape[0], 100, patch_size * patch_size)
    return patches.to(device)

In [ ]:
# === CELL 2b (FIX): BloodMNIST custom loader ===

def load_bloodmnist():
    # make sure the root directory is available
    candidates = [
        Path.cwd() / "data_medmnist",
        Path("/content/data_medmnist"),
        Path.home() / ".medmnist",
        Path("/tmp/medmnist"),
    ]
    root_dir = None
    for p in candidates:
        try:
            p.mkdir(parents=True, exist_ok=True)
            root_dir = str(p)
            break
        except Exception:
            continue
    if root_dir is None:
        raise RuntimeError("Gagal membuat direktori data untuk MedMNIST.")
    os.environ["MEDMNIST_DATA_PATH"] = root_dir

    transform = transforms.Compose([
        transforms.Resize((22, 22)),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])

    # train+val → train_all, test is still test
    train_set = BloodMNIST(root=root_dir, split="train", transform=transform, download=True, as_rgb=False)
    val_set   = BloodMNIST(root=root_dir, split="val",   transform=transform, download=True, as_rgb=False)
    test_set  = BloodMNIST(root=root_dir, split="test",  transform=transform, download=True, as_rgb=False)

    train_all = torch.utils.data.ConcatDataset([train_set, val_set])

    def one_hot_encode(labels, num_classes=NUM_CLASSES):
        return torch.eye(num_classes, device=device)[labels]

    def collate_fn(batch):
        imgs, lbls = zip(*batch)
        imgs = torch.stack(imgs).to(device)

        # label can be scalar or array([k])
        lbls = torch.tensor(
            [int(l) if isinstance(l, (int, np.integer)) else int(l[0]) for l in lbls],
            device=device
        ).long()
        patches = extract_patches(imgs)                       # (B,100,9)
        return patches, one_hot_encode(lbls, num_classes=NUM_CLASSES)

    train_loader = torch.utils.data.DataLoader(
        train_all, batch_size=64, shuffle=True, collate_fn=collate_fn, drop_last=False
    )
    test_loader = torch.utils.data.DataLoader(
        test_set, batch_size=64, shuffle=False, collate_fn=collate_fn, drop_last=False
    )
    return train_loader, test_loader

In [ ]:
# CELL 3 : Quantum Device
# QFE1
n_qubits_qfe1      = 9
ansatz_layers_qfe1 = 2
# QFE2
n_qubits_qfe2      = 9
ansatz_layers_qfe2 = 2
# number of patches per image, must be the same as in CELL 2
n_patches = 100

# QAOA-Heuristic Ansatz with separate RZ and RX parameters
def ansatz(rz_params, rx_params, num_layers, n_qubits, qubit_offset=0):
    """
    rz_params, rx_params: shape (num_layers, n_qubits)
    """
    # make sure to be a PyTorch float32 tensor
    rz = rz_params.clone().detach().to(torch.float32).view(num_layers, n_qubits)
    rx = rx_params.clone().detach().to(torch.float32).view(num_layers, n_qubits)

    for layer in range(num_layers):
        # 1) Hadamard each qubit
        for i in range(qubit_offset, qubit_offset + n_qubits):
            qml.Hadamard(wires=i)

        # 2) Ring-CNOT + RZ + CNOT
        for i in range(qubit_offset, qubit_offset + n_qubits):
            j = qubit_offset + ((i - qubit_offset + 1) % n_qubits)
            qml.CNOT(wires=[i, j])
            qml.RZ(rz[layer, i-qubit_offset], wires=j)
            qml.CNOT(wires=[i, j])

        # 3) RX
        for i in range(qubit_offset, qubit_offset + n_qubits):
            qml.RX(rx[layer, i-qubit_offset], wires=i)


# ─ Quantum Devices ─────────────────────────────────────────────────────────────
dev1 = qml.device("default.qubit", wires=n_qubits_qfe1)
dev2 = qml.device("default.qubit", wires=n_qubits_qfe2)

# QFE1: measure Z on each qubit → 9 observables
@qml.qnode(dev1, interface="torch", diff_method="best")
def qnode_qfe1(inputs, weights_rz, weights_rx):
    inputs     = torch.as_tensor(inputs,  dtype=torch.float32)
    weights_rz = torch.as_tensor(weights_rz, dtype=torch.float32)
    weights_rx = torch.as_tensor(weights_rx, dtype=torch.float32)

    qml.AngleEmbedding(inputs, wires=range(n_qubits_qfe1), rotation="Y")
    ansatz(weights_rz.view(ansatz_layers_qfe1, n_qubits_qfe1),
           weights_rx.view(ansatz_layers_qfe1, n_qubits_qfe1),
           num_layers=ansatz_layers_qfe1,
           n_qubits=n_qubits_qfe1)
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits_qfe1)]

# QFE2: measure Z on each qubit → 9 observables
@qml.qnode(dev2, interface="torch", diff_method="best")
def qnode_qfe2(inputs, weights_rz, weights_rx):
    inputs     = torch.as_tensor(inputs,  dtype=torch.float32)
    weights_rz = torch.as_tensor(weights_rz, dtype=torch.float32)
    weights_rx = torch.as_tensor(weights_rx, dtype=torch.float32)

    qml.AngleEmbedding(inputs, wires=range(n_qubits_qfe2), rotation="Y")
    ansatz(weights_rz.view(ansatz_layers_qfe2, n_qubits_qfe2),
           weights_rx.view(ansatz_layers_qfe2, n_qubits_qfe2),
           num_layers=ansatz_layers_qfe2,
           n_qubits=n_qubits_qfe2)
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits_qfe2)]


# ─ Feature Extractor QFE1 ───────────────────────────────────────────────────────────
class QFE1Extractor(nn.Module):
    def __init__(self):
        super().__init__()
        weight_shapes = {
            "weights_rz": (ansatz_layers_qfe1, n_qubits_qfe1),
            "weights_rx": (ansatz_layers_qfe1, n_qubits_qfe1),
        }
        self.q_layer = qml.qnn.TorchLayer(qnode_qfe1, weight_shapes)

        self.project = nn.Linear(n_qubits_qfe1 * n_patches, 18).to(device)
        self.pool1   = nn.MaxPool1d(kernel_size=2, stride=2)

    def forward(self, x):
        b, n_p, p = x.shape
        flat = x.view(b * n_p, p)      # (b*100,9)
        feats = self.q_layer(flat)     # (b*100,9)
        feats = feats.view(b, n_p * feats.shape[1])  # → (b,900)
        proj  = torch.tanh(self.project(feats)) * np.pi  # (b,18)
        proj  = proj.unsqueeze(1)                       # (b,1,18)
        pool  = self.pool1(proj).squeeze(1)             # (b,9)
        return pool

# ─ Feature Extractor QFE2 ───────────────────────────────────────────────────────────
class QFE2Extractor(nn.Module):
    def __init__(self):
        super().__init__()
        weight_shapes = {
            "weights_rz": (ansatz_layers_qfe2, n_qubits_qfe2),
            "weights_rx": (ansatz_layers_qfe2, n_qubits_qfe2),
        }
        self.q_layer = qml.qnn.TorchLayer(qnode_qfe2, weight_shapes)
        self.project = nn.Linear(n_qubits_qfe2, 54).to(device)
        self.pool2   = nn.MaxPool1d(kernel_size=2, stride=2)

    def forward(self, x):
        feats = self.q_layer(x)                          # (batch,9)
        proj  = torch.tanh(self.project(feats)) * np.pi  # (batch,54)
        proj  = proj.unsqueeze(1)                        # (batch,1,54)
        pool  = self.pool2(proj).squeeze(1)              # (batch,27)
        return pool                                      # (batch,27)


# Visualizing Quantum Circuits (QFE1 & QFE2)
def draw_quantum_circuit_qfe1():
    w_rz = np.random.randn(ansatz_layers_qfe1, n_qubits_qfe1)
    w_rx = np.random.randn(ansatz_layers_qfe1, n_qubits_qfe1)
    inputs = np.random.rand(n_qubits_qfe1)
    fig, ax = qml.draw_mpl(qnode_qfe1)(inputs, w_rz, w_rx)
    plt.title("QUANTUM FEATURE EXTRACTION (QFE) 1")
    plt.show()

def draw_quantum_circuit_qfe2():
    w_rz = np.random.randn(ansatz_layers_qfe2, n_qubits_qfe2)
    w_rx = np.random.randn(ansatz_layers_qfe2, n_qubits_qfe2)
    inputs = np.random.rand(n_qubits_qfe2)
    fig, ax = qml.draw_mpl(qnode_qfe2)(inputs, w_rz, w_rx)
    plt.title("QUANTUM FEATURE EXTRACTION (QFE) 2")
    plt.show()

# Call visualization
draw_quantum_circuit_qfe1()
draw_quantum_circuit_qfe2()


# ─ QuantumCNN ─────────────────────────────────────────────────────────────────
class QuantumCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.qfe1 = QFE1Extractor()
        self.qfe2 = QFE2Extractor()

        # FC classifier: 27 → 128 → 64 → 8 (Multi-Class for BloodMNIST)
        self.fc1 = nn.Linear(27, 128).to(device)
        self.fc2 = nn.Linear(128, 64).to(device)
        self.fc3 = nn.Linear(64, NUM_CLASSES).to(device)

        for layer in (self.fc1, self.fc2):
            nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
            nn.init.zeros_(layer.bias)

    def forward(self, x):
        # x: (batch,100,9)
        h1 = self.qfe1(x)   # → (batch,9)
        h2 = self.qfe2(h1)  # → (batch,27)
        z  = torch.relu(self.fc1(h2))
        z  = torch.relu(self.fc2(z))
        return self.fc3(z)  # → (batch,2)

model = QuantumCNN().to(device)

train_loader, test_loader = load_bloodmnist()

optimizer = optim.Adam(model.parameters(), lr=0.0005)
loss_fn = nn.CrossEntropyLoss()

# Display model parameter counts
num_params = sum(p.numel() for p in model.parameters())
num_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters: {num_params}")
print(f"Trainable Parameters: {num_trainable_params}")


In [ ]:
# === CELL 4: evaluate() auto-binary/multiclass ===

from sklearn.metrics import roc_auc_score

def _to_idx(y):
    # Support scalar label (shape [B]) or one-hot (shape [B, C])
    return y.argmax(dim=1) if y.dim() > 1 else y

def evaluate(model, data_loader):
    loss_sum, correct, total = 0.0, 0, 0
    y_true, y_pred = [], []
    y_scores_bin = []     # for binary
    y_scores_all = []     # for multiclass (N,C)
    model.eval()
    with torch.no_grad():
        for images, labels in data_loader:
            logits = model(images)               # (B,C)
            lbl    = _to_idx(labels)             # (B,)
            loss   = loss_fn(logits, lbl)
            loss_sum += loss.item()

            probs = torch.softmax(logits, dim=1).cpu().numpy()  # (B,C)
            preds = logits.argmax(dim=1).cpu().numpy()
            trues = lbl.cpu().numpy()

            if NUM_CLASSES == 2:
                y_scores_bin.extend(probs[:, 1].tolist())       # prob positive class
            else:
                y_scores_all.append(probs)

            y_pred.extend(preds.tolist())
            y_true.extend(trues.tolist())

            correct += (preds == trues).sum()
            total   += len(trues)

    acc  = correct / total
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score   (y_true, y_pred, average='macro', zero_division=0)
    f1   = f1_score       (y_true, y_pred, average='macro', zero_division=0)

    # calculate AUC according to the number of classes
    try:
        if NUM_CLASSES == 2:
            auc = roc_auc_score(y_true, y_scores_bin)
        else:
            y_scores = np.vstack(y_scores_all)                  # (N,C)
            auc = roc_auc_score(y_true, y_scores, multi_class='ovr', average='macro')
    except Exception:
        auc = float('nan')

    return loss_sum / len(data_loader), acc, prec, rec, f1, auc, y_true, y_pred

# Iteration-based loop training with progress bar
def train_iterations(model,
                     train_loader,
                     test_loader,
                     max_iterations=1100,       # ← iter must be 5 epochs
                     eval_interval=1100):       # ← evaluation in the final iter

    start_time = time.perf_counter()  # ← high-res start
    fc_time_acc = 0.0
    train_losses, train_accs = [], []
    test_losses, test_accs = [], []
    test_precs, test_recs, test_f1s, test_aucs = [], [], [], []

    loader_iter = iter(train_loader)
    pbar = tqdm(total=max_iterations,
                desc="Iter 0/{}".format(max_iterations),
                ncols=100, leave=True)

    for iteration in range(1, max_iterations + 1):
        pbar.set_description(f"Iter {iteration}/{max_iterations}")

        try:
            images, labels = next(loader_iter)
        except StopIteration:
            loader_iter = iter(train_loader)
            images, labels = next(loader_iter)

        model.train()
        optimizer.zero_grad()

        # 1) Quantum FE
        h1 = model.qfe1(images)     # (B, 9)
        h2 = model.qfe2(h1)         # (B, 27)

        # 2) FC-only timing (FAIR)
        feats = h2.to(device, dtype=torch.float32).contiguous()
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        t0 = time.perf_counter()
        z  = torch.relu(model.fc1(feats))
        z  = torch.relu(model.fc2(z))
        outputs = model.fc3(z)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        fc_time_acc += (t1 - t0)

        # 3) Backprop
        lbl  = _to_idx(labels)
        loss = loss_fn(outputs, lbl)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        batch_acc = (outputs.argmax(1).cpu() == lbl.cpu()).float().mean().item()
        train_accs.append(batch_acc)

        pbar.set_postfix({"Loss": f"{loss.item():.4f}",
                          "Acc":  f"{batch_acc:.4f}"})
        pbar.update(1)

        if iteration % eval_interval == 0 or iteration == max_iterations:
            (test_loss, test_acc, test_prec,
             test_rec, test_f1, test_auc, y_true, y_pred) = evaluate(model, test_loader)

            test_losses.append(test_loss)
            test_accs.append(test_acc)
            test_precs.append(test_prec)
            test_recs.append(test_rec)
            test_f1s.append(test_f1)
            test_aucs.append(test_auc)

            tqdm.write(
                f"→ Eval @Iter {iteration}: "
                f"Test Loss={test_loss:.4f}, Test Acc={test_acc:.4f}, "
                f"Prec={test_prec:.4f}, Rec={test_rec:.4f}, F1={test_f1:.4f}, "
                f"AUC={test_auc:.4f}"
            )

            # Plot confusion matrix
            cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
            disp = ConfusionMatrixDisplay(cm, display_labels=list(range(NUM_CLASSES)))
            fig, ax = plt.subplots(figsize=(6,6))
            disp.plot(ax=ax, cmap='Blues', colorbar=False)
            ax.set_title(f"QFE - BloodMNIST")  # Confusion Matrix
            plt.show()

    pbar.close()
    total_time = time.perf_counter() - start_time  # ← high-res total
    print(f"\nTotal training time: {total_time:.2f}s")
    print(f"Total FC-Network computation time: {fc_time_acc:.2f}s")

    # graph axis
    x_iter = list(range(1, max_iterations + 1))
    x_test = list(range(eval_interval, max_iterations+1, eval_interval))
    if x_test and x_test[-1] != max_iterations:
        x_test.append(max_iterations)

    # Graph 1: Train Loss vs Iteration
    plt.figure(figsize=(6,4))
    plt.plot(x_iter, train_losses, label='Train Loss')
    plt.plot(x_test, test_losses, label='Test Loss')
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.ylim(0, 3)
    plt.legend()
    plt.title('Loss vs Iteration')
    plt.show()

    # Graph 2: Train & Test Accuracy vs Iteration
    plt.figure(figsize=(6,4))
    plt.plot(x_iter, train_accs, label='Train Acc')
    plt.plot(x_test, test_accs, label='Test Acc')
    plt.xlabel('Iteration')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1)
    plt.legend()
    plt.title('Accuracy vs Iteration')
    plt.show()

    # Graph 3: All TRAIN (loss & acc)
    train_loss_color = 'tab:orange'
    train_acc_color  = 'tab:green'
    fig, ax1 = plt.subplots(figsize=(6,4))
    ax2 = ax1.twinx()
    ax1.plot(x_iter, train_losses, color=train_loss_color, label='Batch Loss')
    ax2.plot(x_iter, train_accs,  color=train_acc_color,  label='Batch Acc')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Loss')
    ax2.set_ylabel('Accuracy')
    ax1.set_ylim(0, 3)
    ax2.set_ylim(0, 1)
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
    plt.title('Train Performance')
    plt.show()

    # Graph 4: All TEST (loss & acc)
    test_loss_color = 'tab:orange'
    test_acc_color  = 'tab:green'
    fig, ax1 = plt.subplots(figsize=(6,4))
    ax2 = ax1.twinx()
    ax1.plot(x_test, test_losses, color=test_loss_color, label='Test Loss')
    ax2.plot(x_test, test_accs,  color=test_acc_color,  label='Test Acc')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Loss')
    ax2.set_ylabel('Accuracy')
    ax1.set_ylim(0, 3)
    ax2.set_ylim(0, 1)
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
    plt.title('Test Performance')
    plt.show()

# Call training
train_iterations(model, train_loader, test_loader,
                 max_iterations=1100,
                 eval_interval=1100)

In [ ]:
# CELL 5 : Saving the weights of the trained model
def save_model(model, filename="QFE_BloodMNIST.pth"):
    torch.save(model.state_dict(), filename)
    print(f"Model state_dict saved to {filename}")

# After the training is finished
save_model(model)